# Pydantic과 Anthropic tool use를 사용하는 메모 저장 tool

이 예제에서는 작성자와 metadata가 있는 메모를 저장하는 tool을 만들고 tool 호출 시 model 응답을 Pydantic으로 검증합니다. 필요한 Pydantic model을 정의하고 tool 호출을 처리하며 model 응답이 기대 schema를 따르는지 확인합니다.


## 1단계: environment 설정하기
먼저 필요한 library를 설치하고 Claude API client를 설정하겠습니다.


In [ ]:
%pip install anthropic pydantic 'pydantic[email]'

In [2]:
from anthropic import Anthropic
from pydantic import BaseModel, EmailStr, Field

client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

## 2단계: Pydantic model 정의하기

메모, 작성자, model 응답의 기대 schema를 표현할 Pydantic model을 정의합니다. 메모 저장 시 model 응답을 검증하고 type-check할 수 있습니다.


In [18]:
class Author(BaseModel):
    name: str
    email: EmailStr


class Note(BaseModel):
    note: str
    author: Author
    tags: list[str] | None = None
    priority: int = Field(ge=1, le=5, default=3)
    is_public: bool = False


class SaveNoteResponse(BaseModel):
    success: bool
    message: str

## 3단계: client 측 tool 정의하기

다음으로 chatbot이 메모 저장에 사용할 client 측 tool을 정의합니다.


In [17]:
tools = [
    {
        "name": "save_note",
        "description": "A tool that saves a note with the author and metadata.",
        "input_schema": {
            "type": "object",
            "properties": {
                "note": {"type": "string", "description": "The content of the note to be saved."},
                "author": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "The name of the author."},
                        "email": {
                            "type": "string",
                            "format": "email",
                            "description": "The email address of the author.",
                        },
                    },
                    "required": ["name", "email"],
                },
                "priority": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 5,
                    "default": 3,
                    "description": "The priority level of the note (1-5).",
                },
                "is_public": {
                    "type": "boolean",
                    "default": False,
                    "description": "Indicates whether the note is publicly accessible.",
                },
            },
            "required": ["note", "author"],
        },
    }
]

## 4단계: 메모 저장 tool 구현하기
메모를 성공적으로 저장했다는 출력만 하는 dummy 저장 함수를 만듭니다. 실제로 어딘가에 저장하려면 이 함수를 구현할 수 있습니다.


In [23]:
def save_note(note: str, author: dict, priority: int = 3, is_public: bool = False) -> None:
    print("Note saved successfully!")

## 5단계: tool 호출 처리하고 응답 생성하기
Claude의 tool 호출을 처리하고 메모 저장 성공을 나타내는 응답을 생성하는 함수를 만듭니다.


In [24]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "save_note":
        note = Note(
            note=tool_input["note"],
            author=Author(name=tool_input["author"]["name"], email=tool_input["author"]["email"]),
            priority=tool_input.get("priority", 3),
            is_public=tool_input.get("is_public", False),
        )
        save_note(note.note, note.author.model_dump(), note.priority, note.is_public)
        return SaveNoteResponse(success=True, message="Note saved successfully!")


def generate_response(save_note_response):
    return f"Response: {save_note_response.message}"

## 6단계: chatbot과 상호 작용하기

이제 chatbot과 상호 작용하는 함수를 만듭니다. user message를 보내고, Claude의 tool 호출을 처리하고, 응답을 생성하고, Pydantic으로 model 응답을 검증하고, 최종 응답을 사용자에게 반환합니다.


In [21]:
def chatbot_interaction(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages = [{"role": "user", "content": user_message}]

    message = client.messages.create(
        model=MODEL_NAME, max_tokens=4096, tools=tools, messages=messages
    )

    print("\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    if message.stop_reason == "tool_use":
        tool_use = next(block for block in message.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print(f"Tool Input: {tool_input}")

        save_note_response = process_tool_call(tool_name, tool_input)

        print(f"Tool Result: {save_note_response}")

        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            messages=[
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": message.content},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": str(save_note_response),
                        }
                    ],
                },
            ],
            tools=tools,
        )
    else:
        response = message

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )
    print(response.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

## 7단계: chatbot 테스트하기
메모를 저장하는 예제 query로 chatbot을 테스트하겠습니다.


In [26]:
chatbot_interaction("""
Can you save a private note with the following details?
Note: Remember to buy milk and eggs.
Author: John Doe (johndoe@gmail.com)
Priority: 4
""")


User Message: 
Can you save a private note with the following details?
Note: Remember to buy milk and eggs.
Author: John Doe (johndoe@gmail.com)
Priority: 4


Initial Response:
Stop Reason: tool_use
Content: [ContentBlock(text='<thinking>\nThe relevant tool to use here is save_note, as the request is to save a note with specific details.\n\nLet\'s go through the parameters one-by-one:\n\nnote: The user provided the note content: "Remember to buy milk and eggs."\nauthor: The user provided the author details: \n{\n  "name": "John Doe",\n  "email": "johndoe@gmail.com"\n}\nis_public: While the user didn\'t explicitly specify, they asked for a "private note", so we can infer is_public should be false.\npriority: The user specified a priority of 4.\n\nAll the required parameters have been provided or can be reasonably inferred from the request. We have enough information to make the save_note call.\n</thinking>', type='text'), ContentBlockToolUse(id='toolu_015iteV2eC1C7aUodbkotfiS', input={

'Your private note has been saved successfully with the following details:\n\nNote: Remember to buy milk and eggs. \nAuthor: John Doe (johndoe@gmail.com)\nPriority: 4\nVisibility: Private\n\nPlease let me know if you need anything else!'

이 예제에서는 작성자와 metadata가 있는 메모를 저장하는 tool을 만들었습니다. chatbot은 save_note tool로 메모를 저장하고, tool 호출 시 Pydantic으로 model 응답을 검증합니다. Note, Author, SaveNoteResponse model은 tool 입력과 model 응답이 기대 schema를 따르게 합니다.

명확한 Pydantic model을 정의하고 model 응답 검증에 사용하면 chatbot과 상호 작용하고 메모를 저장할 때 신뢰성과 안전성 계층이 추가됩니다.
